In [ ]:
#import library
import re
import math
import pandas as pd

from Sastrawi.StopWordRemover.StopWordRemoverFactory import StopWordRemoverFactory
from sklearn.feature_extraction.text import TfidfVectorizer

In [ ]:
#dataset 4 dokumen
dokumen = [
    "Sistem komputer mengolah data menggunakan perangkat keras dan perangkat lunak.",
    "Jaringan komputer menghubungkan perangkat untuk pertukaran data dan informasi.",
    "Kecerdasan buatan menggunakan data untuk menghasilkan prediksi dan rekomendasi.",
    "Sistem temu kembali mencari dokumen berdasarkan kebutuhan informasi pengguna."
]

for i, doc in enumerate(dokumen, 1):
    print(f"Dokumen {i}: {doc}")

Dokumen 1: Sistem komputer mengolah data menggunakan perangkat keras dan perangkat lunak.
Dokumen 2: Jaringan komputer menghubungkan perangkat untuk pertukaran data dan informasi.
Dokumen 3: Kecerdasan buatan menggunakan data untuk menghasilkan prediksi dan rekomendasi.
Dokumen 4: Sistem temu kembali mencari dokumen berdasarkan kebutuhan informasi pengguna.


In [ ]:
# Preprocessing dokumen
stopword_factory = StopWordRemoverFactory()
stopwords = set(stopword_factory.get_stop_words())

def preprocessing_sederhana(text):
    # Case folding
    text = text.lower()

    # Cleaning
    text = re.sub(r'[^a-zA-Z\s]', ' ', text)
    text = re.sub(r'\s+', ' ', text).strip()

    # Tokenisasi
    tokens = text.split()

    # Stopwords removal
    tokens = [token for token in tokens if token not in stopwords]

    return tokens

tokens_dokumen = [preprocessing_sederhana(doc) for doc in dokumen]

for i, tokens in enumerate(tokens_dokumen, 1):
    print(f"Dokumen {i}: {tokens}")

Dokumen 1: ['sistem', 'komputer', 'mengolah', 'data', 'menggunakan', 'perangkat', 'keras', 'perangkat', 'lunak']
Dokumen 2: ['jaringan', 'komputer', 'menghubungkan', 'perangkat', 'pertukaran', 'data', 'informasi']
Dokumen 3: ['kecerdasan', 'buatan', 'menggunakan', 'data', 'menghasilkan', 'prediksi', 'rekomendasi']
Dokumen 4: ['sistem', 'temu', 'mencari', 'dokumen', 'berdasarkan', 'kebutuhan', 'informasi', 'pengguna']


In [ ]:
#bag of words
vocabulary = sorted(
    set(token for tokens in tokens_dokumen for token in tokens)
)

bow = []

for tokens in tokens_dokumen:
    row = []
    for term in vocabulary:
        row.append(tokens.count(term))
    bow.append(row)

bow_df = pd.DataFrame(
    bow,
    columns=vocabulary,
    index=[f"Dokumen {i}" for i in range(1, 5)]
)

bow_df

,berdasarkan,buatan,data,dokumen,informasi,jaringan,kebutuhan,kecerdasan,keras,komputer,...,menghasilkan,menghubungkan,mengolah,pengguna,perangkat,pertukaran,prediksi,rekomendasi,sistem,temu
Dokumen 1,0,0,1,0,0,0,0,0,1,1,...,0,0,1,0,2,0,0,0,1,0
Dokumen 2,0,0,1,0,1,1,0,0,0,1,...,0,1,0,0,1,1,0,0,0,0
Dokumen 3,0,1,1,0,0,0,0,1,0,0,...,1,0,0,0,0,0,1,1,0,0
Dokumen 4,1,0,0,1,1,0,1,0,0,0,...,0,0,0,1,0,0,0,0,1,1


In [ ]:
#term frequency
tf_data = []

for tokens in tokens_dokumen:
    total_token = len(tokens)
    row = {}

    for term in vocabulary:
        count = tokens.count(term)
        row[term] = count / total_token if total_token > 0 else 0

    tf_data.append(row)

tf_df = pd.DataFrame(
    tf_data,
    columns=vocabulary,
    index=[f"Dokumen {i}" for i in range(1, 5)]
)

tf_df


,berdasarkan,buatan,data,dokumen,informasi,jaringan,kebutuhan,kecerdasan,keras,komputer,...,menghasilkan,menghubungkan,mengolah,pengguna,perangkat,pertukaran,prediksi,rekomendasi,sistem,temu
Dokumen 1,0.000,0.000000,0.111111,0.000,0.000000,0.000000,0.000,0.000000,0.111111,0.111111,...,0.000000,0.000000,0.111111,0.000,0.222222,0.000000,0.000000,0.000000,0.111111,0.000
Dokumen 2,0.000,0.000000,0.142857,0.000,0.142857,0.142857,0.000,0.000000,0.000000,0.142857,...,0.000000,0.142857,0.000000,0.000,0.142857,0.142857,0.000000,0.000000,0.000000,0.000
Dokumen 3,0.000,0.142857,0.142857,0.000,0.000000,0.000000,0.000,0.142857,0.000000,0.000000,...,0.142857,0.000000,0.000000,0.000,0.000000,0.000000,0.142857,0.142857,0.000000,0.000
Dokumen 4,0.125,0.000000,0.000000,0.125,0.125000,0.000000,0.125,0.000000,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.125,0.000000,0.000000,0.000000,0.000000,0.125000,0.125


In [ ]:
#document frequency
N = len(dokumen)

df_data = {}

for term in vocabulary:
    df_data[term] = sum(
        1 for tokens in tokens_dokumen
        if term in tokens
    )

# Rumus IDF manual:
# IDF = log(N / DF)

idf_data = {
    term: math.log(N / df_data[term])
    for term in vocabulary
}

df_idf = pd.DataFrame({
    "Term": vocabulary,
    "DF": [df_data[term] for term in vocabulary],
    "IDF": [idf_data[term] for term in vocabulary]
})

df_idf

,Term,DF,IDF
0,berdasarkan,1,1.386294
1,buatan,1,1.386294
2,data,3,0.287682
3,dokumen,1,1.386294
4,informasi,2,0.693147
5,jaringan,1,1.386294
6,kebutuhan,1,1.386294
7,kecerdasan,1,1.386294
8,keras,1,1.386294
9,komputer,2,0.693147


In [ ]:
#tf-idf manual
tfidf_manual = []

for i in range(N):
    row = {}

    for term in vocabulary:
        row[term] = tf_df.iloc[i][term] * idf_data[term]

    tfidf_manual.append(row)

tfidf_manual_df = pd.DataFrame(
    tfidf_manual,
    columns=vocabulary,
    index=[f"Dokumen {i}" for i in range(1, 5)]
)

tfidf_manual_df


,berdasarkan,buatan,data,dokumen,informasi,jaringan,kebutuhan,kecerdasan,keras,komputer,...,menghasilkan,menghubungkan,mengolah,pengguna,perangkat,pertukaran,prediksi,rekomendasi,sistem,temu
Dokumen 1,0.000000,0.000000,0.031965,0.000000,0.000000,0.000000,0.000000,0.000000,0.154033,0.077016,...,0.000000,0.000000,0.154033,0.000000,0.154033,0.000000,0.000000,0.000000,0.077016,0.000000
Dokumen 2,0.000000,0.000000,0.041097,0.000000,0.099021,0.198042,0.000000,0.000000,0.000000,0.099021,...,0.000000,0.198042,0.000000,0.000000,0.099021,0.198042,0.000000,0.000000,0.000000,0.000000
Dokumen 3,0.000000,0.198042,0.041097,0.000000,0.000000,0.000000,0.000000,0.198042,0.000000,0.000000,...,0.198042,0.000000,0.000000,0.000000,0.000000,0.000000,0.198042,0.198042,0.000000,0.000000
Dokumen 4,0.173287,0.000000,0.000000,0.173287,0.086643,0.000000,0.173287,0.000000,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.173287,0.000000,0.000000,0.000000,0.000000,0.086643,0.173287


In [ ]:
#tf-idf sklearn
dokumen_preprocessed = [
    " ".join(tokens)
    for tokens in tokens_dokumen
]

vectorizer = TfidfVectorizer(
    tokenizer=str.split,
    preprocessor=None,
    token_pattern=None,
    lowercase=False,
    norm=None,
    smooth_idf=False
)

tfidf_sklearn = vectorizer.fit_transform(dokumen_preprocessed)

tfidf_sklearn_df = pd.DataFrame(
    tfidf_sklearn.toarray(),
    columns=vectorizer.get_feature_names_out(),
    index=[f"Dokumen {i}" for i in range(1, 5)]
)

tfidf_sklearn_df

,berdasarkan,buatan,data,dokumen,informasi,jaringan,kebutuhan,kecerdasan,keras,komputer,...,menghasilkan,menghubungkan,mengolah,pengguna,perangkat,pertukaran,prediksi,rekomendasi,sistem,temu
Dokumen 1,0.000000,0.000000,1.287682,0.000000,0.000000,0.000000,0.000000,0.000000,2.386294,1.693147,...,0.000000,0.000000,2.386294,0.000000,3.386294,0.000000,0.000000,0.000000,1.693147,0.000000
Dokumen 2,0.000000,0.000000,1.287682,0.000000,1.693147,2.386294,0.000000,0.000000,0.000000,1.693147,...,0.000000,2.386294,0.000000,0.000000,1.693147,2.386294,0.000000,0.000000,0.000000,0.000000
Dokumen 3,0.000000,2.386294,1.287682,0.000000,0.000000,0.000000,0.000000,2.386294,0.000000,0.000000,...,2.386294,0.000000,0.000000,0.000000,0.000000,0.000000,2.386294,2.386294,0.000000,0.000000
Dokumen 4,2.386294,0.000000,0.000000,2.386294,1.693147,0.000000,2.386294,0.000000,0.000000,0.000000,...,0.000000,0.000000,0.000000,2.386294,0.000000,0.000000,0.000000,0.000000,1.693147,2.386294


In [ ]:
# Perbandingan TF-IDF Manual dan Scikit-Learn
print("TF-IDF MANUAL")
display(tfidf_manual_df)

print("\nTF-IDF SCIKIT-LEARN")
display(tfidf_sklearn_df)

print("\nSELISIH ABSOLUT")
selisih = (tfidf_manual_df - tfidf_sklearn_df).abs()
display(selisih)

TF-IDF MANUAL


,berdasarkan,buatan,data,dokumen,informasi,jaringan,kebutuhan,kecerdasan,keras,komputer,...,menghasilkan,menghubungkan,mengolah,pengguna,perangkat,pertukaran,prediksi,rekomendasi,sistem,temu
Dokumen 1,0.000000,0.000000,0.031965,0.000000,0.000000,0.000000,0.000000,0.000000,0.154033,0.077016,...,0.000000,0.000000,0.154033,0.000000,0.154033,0.000000,0.000000,0.000000,0.077016,0.000000
Dokumen 2,0.000000,0.000000,0.041097,0.000000,0.099021,0.198042,0.000000,0.000000,0.000000,0.099021,...,0.000000,0.198042,0.000000,0.000000,0.099021,0.198042,0.000000,0.000000,0.000000,0.000000
Dokumen 3,0.000000,0.198042,0.041097,0.000000,0.000000,0.000000,0.000000,0.198042,0.000000,0.000000,...,0.198042,0.000000,0.000000,0.000000,0.000000,0.000000,0.198042,0.198042,0.000000,0.000000
Dokumen 4,0.173287,0.000000,0.000000,0.173287,0.086643,0.000000,0.173287,0.000000,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.173287,0.000000,0.000000,0.000000,0.000000,0.086643,0.173287



TF-IDF SCIKIT-LEARN


,berdasarkan,buatan,data,dokumen,informasi,jaringan,kebutuhan,kecerdasan,keras,komputer,...,menghasilkan,menghubungkan,mengolah,pengguna,perangkat,pertukaran,prediksi,rekomendasi,sistem,temu
Dokumen 1,0.000000,0.000000,1.287682,0.000000,0.000000,0.000000,0.000000,0.000000,2.386294,1.693147,...,0.000000,0.000000,2.386294,0.000000,3.386294,0.000000,0.000000,0.000000,1.693147,0.000000
Dokumen 2,0.000000,0.000000,1.287682,0.000000,1.693147,2.386294,0.000000,0.000000,0.000000,1.693147,...,0.000000,2.386294,0.000000,0.000000,1.693147,2.386294,0.000000,0.000000,0.000000,0.000000
Dokumen 3,0.000000,2.386294,1.287682,0.000000,0.000000,0.000000,0.000000,2.386294,0.000000,0.000000,...,2.386294,0.000000,0.000000,0.000000,0.000000,0.000000,2.386294,2.386294,0.000000,0.000000
Dokumen 4,2.386294,0.000000,0.000000,2.386294,1.693147,0.000000,2.386294,0.000000,0.000000,0.000000,...,0.000000,0.000000,0.000000,2.386294,0.000000,0.000000,0.000000,0.000000,1.693147,2.386294



SELISIH ABSOLUT


,berdasarkan,buatan,data,dokumen,informasi,jaringan,kebutuhan,kecerdasan,keras,komputer,...,menghasilkan,menghubungkan,mengolah,pengguna,perangkat,pertukaran,prediksi,rekomendasi,sistem,temu
Dokumen 1,0.000000,0.000000,1.255717,0.000000,0.000000,0.000000,0.000000,0.000000,2.232262,1.616131,...,0.000000,0.000000,2.232262,0.000000,3.232262,0.000000,0.000000,0.000000,1.616131,0.000000
Dokumen 2,0.000000,0.000000,1.246585,0.000000,1.594126,2.188252,0.000000,0.000000,0.000000,1.594126,...,0.000000,2.188252,0.000000,0.000000,1.594126,2.188252,0.000000,0.000000,0.000000,0.000000
Dokumen 3,0.000000,2.188252,1.246585,0.000000,0.000000,0.000000,0.000000,2.188252,0.000000,0.000000,...,2.188252,0.000000,0.000000,0.000000,0.000000,0.000000,2.188252,2.188252,0.000000,0.000000
Dokumen 4,2.213008,0.000000,0.000000,2.213008,1.606504,0.000000,2.213008,0.000000,0.000000,0.000000,...,0.000000,0.000000,0.000000,2.213008,0.000000,0.000000,0.000000,0.000000,1.606504,2.213008


In [ ]:
# Menampilkan Term dengan Nilai Tertinggi pada Setiap Dokumen
for i in range(len(tfidf_manual_df)):
    dokumen_tabel = tfidf_manual_df.iloc[i]

    term_tertinggi = dokumen_tabel.idxmax()
    nilai_tertinggi = dokumen_tabel.max()

    print(
        f"Dokumen {i + 1}: "
        f"'{term_tertinggi}' "
        f"= {nilai_tertinggi:.4f}"
    )

Dokumen 1: 'keras' = 0.1540
Dokumen 2: 'jaringan' = 0.1980
Dokumen 3: 'buatan' = 0.1980
Dokumen 4: 'berdasarkan' = 0.1733
